In [39]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer # dataset
from sklearn.model_selection import train_test_split # train/test lib
from sklearn.preprocessing import StandardScaler # scale data
from sklearn.linear_model import LogisticRegression # run logistic 
from sklearn.metrics import confusion_matrix # use confusion matrix lib

In [2]:
# load dataset
data = load_breast_cancer()


In [3]:
# inspect data
type(data)

sklearn.utils._bunch.Bunch

In [4]:
data.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

From looking at the data, the X/input features is the 'data' key and the Y/features is the 'target' key

X -> model -> $\hat{y}$

During training, we compare the prediction $\hat{y}$ against the actual answer y


In [5]:
# assign x and y and inspect shape
X = data.data
Y = data.target
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

X shape: (569, 30)
Y shape: (569,)


In [6]:
# look at the columns
print(len(data.feature_names))
print(data.feature_names)


30
['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


In [7]:
print(data.target_names)

['malignant' 'benign']


In [8]:
print(X[0])


[1.799e+01 1.038e+01 1.228e+02 1.001e+03 1.184e-01 2.776e-01 3.001e-01
 1.471e-01 2.419e-01 7.871e-02 1.095e+00 9.053e-01 8.589e+00 1.534e+02
 6.399e-03 4.904e-02 5.373e-02 1.587e-02 3.003e-02 6.193e-03 2.538e+01
 1.733e+01 1.846e+02 2.019e+03 1.622e-01 6.656e-01 7.119e-01 2.654e-01
 4.601e-01 1.189e-01]


In [9]:
print(Y[0])

0


In [10]:
# need to figure out how to use a Series to attach X[0] with the feature names
feat_names = pd.Series(data=X[0],index=data.feature_names)
feat_names

mean radius                  17.990000
mean texture                 10.380000
mean perimeter              122.800000
mean area                  1001.000000
mean smoothness               0.118400
mean compactness              0.277600
mean concavity                0.300100
mean concave points           0.147100
mean symmetry                 0.241900
mean fractal dimension        0.078710
radius error                  1.095000
texture error                 0.905300
perimeter error               8.589000
area error                  153.400000
smoothness error              0.006399
compactness error             0.049040
concavity error               0.053730
concave points error          0.015870
symmetry error                0.030030
fractal dimension error       0.006193
worst radius                 25.380000
worst texture                17.330000
worst perimeter             184.600000
worst area                 2019.000000
worst smoothness              0.162200
worst compactness        

In [11]:
# how many malignant and benign do we have?
Y_mal_mask = (Y == 0)
num_malignant = Y_mal_mask.sum()

Y_benign_mask = (Y == 1)
num_benign = Y_benign_mask.sum()

total_cases = len(data.feature_names)
print(total_cases)

print(f"Number of malignant results: {num_malignant}")
print(f"Number of benign results: {num_benign}")

30
Number of malignant results: 212
Number of benign results: 357


In [12]:
# get benign/ mal percentages
benign_percent = (num_benign / X.shape[0]) * 100
print(f"Benign: {benign_percent:.2f}%")

malignant_percent = (num_malignant / X.shape[0]) * 100
print(f"Malignant: {malignant_percent:.2f}%")


Benign: 62.74%
Malignant: 37.26%


Since we have a naive baseline accuracy of 62.7%, any meaningful model must beat that

Now that we have the numerical representation of the data, we must split the data into training and testing data. 
The model never sees the testing data during training
We will use 80 percent for training data and 20 percent for testing

In [13]:
# use sklearn import train_test_split()
# returns list containing train-test split of inputs
# stratify parameter will preserve 0/1 ratio present in original dataset

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)


In [14]:
# check shape of the 4 variables
X_train_shape = X_train.shape
X_test_shape = X_test.shape
Y_train_shape = Y_train.shape
Y_test_shape = Y_test.shape
print(f"X_train shape: {X_train_shape}")
print(f"X_test shape: {X_test_shape}")
print(f"Y_train shape: {Y_train_shape}")
print(f"Y_test shape: {Y_test_shape}")

X_train shape: (455, 30)
X_test shape: (114, 30)
Y_train shape: (455,)
Y_test shape: (114,)


In [15]:
# get the malignant and benign results from the training and testing data

Y_train_benign = (Y_train == 1).sum()
Y_test_benign = (Y_test == 1).sum()
Y_train_malignant = (Y_train == 0).sum()
Y_test_malignant = (Y_test == 0).sum()

print(f"Y_train benign: {Y_train_benign}")
print(f"Y_test benign: {Y_test_benign}")
print(f"Y_train malignant: {Y_train_malignant}")
print(f"Y_test malignant: {Y_test_malignant}")


Y_train benign: 285
Y_test benign: 72
Y_train malignant: 170
Y_test malignant: 42


After checking the percentages of malignant and benign, the stratify parameter appears to have worked because the distribution was about the same.

Next step is to do feature scaling. Since the values of the different features had a large variance, we will normalize/standardize and run logistic regression

Model sequence:
X_train -------> calculate mean/std -------> scale X_train

X_test -----> scale X_test

In [16]:
# scale the data

scaler = StandardScaler()
scaler.fit(X_train) # fit the training data by getting the mean and std of training data

X_train_scaled = scaler.transform(X_train) 
X_test_scaled = scaler.transform(X_test)

In [17]:
# the mean and std should be close to 0 and 1
X_train_scaled[:,0].mean()

np.float64(-4.337433952909183e-15)

In [18]:
X_train_scaled[:,0].std()

np.float64(1.0)

In [19]:
# build and train the classifier 

model = LogisticRegression()
model.fit(X_train_scaled, Y_train) # takes the inputs/features and correct answers/labels

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [34]:
# now we see if the model can predict things it has never seen

Y_predictions = model.predict(X_test_scaled)

In [35]:
# compare predictions and labels
y_pred = Y_predictions
y_lab = Y_test

print(f"Y prediction values: {y_pred}")
print(f"Y label values: {y_lab}")

""" y_pred are the values the machine generated after we fed it the scaled data.
    y_test are the actual label (true data) that came with original dataset 
"""

Y prediction values: [0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1
 1 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 0 1 1 1 1 1 1 1 0
 0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1
 0 1 1]
Y label values: [0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1
 1 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1 1 0 0 1 1 1 1 1 0 0 1 1 1 1 1 1 1 0
 0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1
 0 1 1]


' y_pred are the values the machine generated after we fed it the scaled data.\n    y_test are the actual label (true data) that came with original dataset \n'

In [38]:
# calculate the accuracy : num correct pred/ total pred

correct_pred = (y_pred == y_lab).sum()
total_pred = len(y_pred)
accuracy = correct_pred/total_pred
accuracy_per = accuracy * 100
print(f"Accuracy: {accuracy_per:.2f}%")

Accuracy: 98.25%


In [40]:
# create confusion matrix to investigate the mistakes

confusion_matx = confusion_matrix(y_lab, y_pred)
confusion_matx

array([[41,  1],
       [ 1, 71]])

### Confusion Matrix Results

The confusion matrix displays the following:

| | Predicted Malignant (0) | Predicted Benign (1) |
|---|---|---|
| Actual Malignant (0) | TP = 41 | FN = 1 |
| Actual Benign (1) | FP = 1 | TN = 71 |

### Interpretation

- **TP = 41:** The model correctly identified 41 malignant tumors.
- **TN = 71:** The model correctly identified 71 benign tumors.
- **FP = 1:** The model incorrectly classified one benign tumor as malignant.
- **FN = 1:** The model incorrectly classified one malignant tumor as benign.

### Calculate Recall for malignant tumors

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

In [51]:
TP = confusion_matx[0, 0]
TN = confusion_matx[1, 1]
FP = confusion_matx[1, 0]
FN = confusion_matx[0, 1]

In [52]:
recall = (TP / (TP + FN)) * 100
print(f"Recall: {recall:.2f}%")
print(f"Of the 42 malignant tumors, {recall:.2f}% were correctly identified")

Recall: 97.62%
Of the 42 malignant tumors, 97.62% were correctly identified


### Calculate precision for malignant tumors

$$
\text {Precision} = \frac{TP}{TP + FP}
$$

In [54]:
precision = (TP / (TP + FP)) * 100
print(f"Precision: {precision:.2f}%")
print("Out of all the tumors we predicted were malignant, approx 97% actually were")

Precision: 97.62%
Out of all the tumors we predicted were malignant, approx 97% actually were


### Calculate F1 Score

$$
\text{F1} = 2 \cdot \frac{precision \cdot recall}{precision + recall}
$$

In [58]:
F1 = 2 * ((precision * recall) / (precision + recall))
print(f"F1 score: {F1:.2f}%")

F1 score: 97.62%
